# ACT IV Ablations Notebook (Colab): Real Trained Model + Delta A/B/C + Cost-Pareto

This notebook runs the current repo scripts on Colab GPU, including real trained inference from:
`training/tenacious_path_b_dpo_lora`.


## Required Inputs

You should have these files in your project directory:

1. `tenacious_bench_v0.1/held_out/tasks.jsonl`
2. `scoring_evaluator.py`
3. `generation_scripts/export_heldout_outputs.py`
4. `generation_scripts/run_act4_ablations.py`
5. `training/tenacious_path_b_dpo_lora/adapter_config.json`
6. `training/tenacious_path_b_dpo_lora/adapter_model.safetensors`

Optional inputs:

1. `training/heldout_baseline_outputs.jsonl` and `training/heldout_prompt_outputs.jsonl` (will be generated if missing)
2. Week 10 retail score for Delta C (`WEEK10_RETAIL_SCORE`)
3. Cost assumptions (`ASSUME_COST_BASELINE`, `ASSUME_COST_PROMPT`, `ASSUME_COST_TRAINED`)


## Stage 1 - Runtime Setup
Run this cell once per new Colab runtime. Then use **Runtime -> Restart runtime**.


In [ ]:
import os

# Keep unsloth + transformers stack consistent on Colab
!pip -q uninstall -y torch torchvision torchaudio xformers unsloth unsloth_zoo transformers trl peft bitsandbytes || true
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Install core libraries without their dependencies (handled by unsloth or explicitly)
!pip -q install --no-deps xformers trl peft accelerate
# Ensure bitsandbytes is installed with its full dependencies at the required version
!pip -q install "bitsandbytes>=0.46.1"

print("Setup complete. Restart runtime now before continuing.")

## Stage 2 - Mount Drive and Set Project Path
Set `PROJECT_DIR` to your repo folder in Drive or local Colab workspace.


In [ ]:
from pathlib import Path
import os

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Example: "/content/drive/MyDrive/tenacious_bench_v0.1"
PROJECT_DIR = Path("/content/drive/MyDrive/tenacious_bench_v0.1")
assert PROJECT_DIR.exists(), f"Project dir not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


## Stage 3 - Validate Required Files


In [ ]:
from pathlib import Path

required = [
    "held_out/tasks.jsonl",
    "scripts/scoring_evaluator.py",
    "scripts/export_heldout_outputs.py",
    "scripts/run_act4_ablations.py",
    "training/tenacious_path_b_dpo_lora/adapter_config.json",
    "training/tenacious_path_b_dpo_lora/adapter_model.safetensors",
]

missing = [p for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing required files:" + " ".join(missing))

print("All required files are present.")


## Stage 4 - (Optional) Generate Baseline and Prompt-Only Outputs
If these files already exist and you want to reuse them, you can skip this stage.


In [ ]:
import subprocess
from pathlib import Path

held_out = "held_out/tasks.jsonl"
baseline_out = Path("training/heldout_baseline_outputs.jsonl")
prompt_out = Path("training/heldout_prompt_outputs.jsonl")

if not baseline_out.exists():
    subprocess.run([
        "python", "scripts/export_heldout_outputs.py",
        "--mode", "baseline",
        "--held-out", held_out,
        "--out", str(baseline_out),
    ], check=True)

if not prompt_out.exists():
    subprocess.run([
        "python", "scripts/export_heldout_outputs.py",
        "--mode", "prompt_only",
        "--held-out", held_out,
        "--out", str(prompt_out),
    ], check=True)

print("baseline:", baseline_out, "exists=", baseline_out.exists())
print("prompt:", prompt_out, "exists=", prompt_out.exists())


## Stage 5 - Export Trained Intervention Outputs (Path B)
This runs the adapter as a rewrite intervention over baseline drafts (`trained_intervene`).


In [ ]:
import subprocess
TRAINED_OUT = "training/heldout_trained_outputs.jsonl"
cmd = [
    "python", "scripts/export_heldout_outputs.py",
    "--mode", "trained_intervene",
    "--held-out", "held_out/tasks.jsonl",
    "--base-outputs-file", "training/heldout_baseline_outputs.jsonl",
    "--adapter-path", "training/tenacious_path_b_dpo_lora",
    "--base-model", "auto",
    "--inference-backend", "auto",
    "--max-seq-length", "1024",
    "--max-new-tokens", "140",
    "--out", TRAINED_OUT,
]
try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print("STDOUT:", result.stdout)
    print("Wrote:", TRAINED_OUT)
except subprocess.CalledProcessError as e:
    print(f"Error executing command: {e}")
    print(f"Command: {e.cmd}")
    print(f"Return code: {e.returncode}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")

## Stage 6 - Smoke Test Ablations (Recommended)


In [ ]:
import subprocess

cmd = [
    "python", "generation_scripts/run_act4_ablations.py",
    "--held-out", "tenacious_bench_v0.1/held_out/tasks.jsonl",
    "--baseline-outputs-file", "training/heldout_baseline_outputs.jsonl",
    "--prompt-outputs-file", "training/heldout_prompt_outputs.jsonl",
    "--trained-outputs-file", "training/heldout_trained_outputs.jsonl",
    "--limit", "10",
    "--bootstrap-iters", "500",
    "--out-ablation", "training/ablation_results_smoke.json",
    "--out-traces", "training/held_out_traces_smoke.jsonl",
]
subprocess.run(cmd, check=True)
print("Smoke run complete.")


## Stage 7 - Full ACT IV Run
Set your optional Week 10 retail score for Delta C. Leave as `None` to skip Delta C.


In [ ]:
import subprocess

WEEK10_RETAIL_SCORE = None  # example: 42.0
ASSUME_COST_BASELINE = 0.0
ASSUME_COST_PROMPT = 0.0
ASSUME_COST_TRAINED = 0.0

cmd = [
    "python", "generation_scripts/run_act4_ablations.py",
    "--held-out", "tenacious_bench_v0.1/held_out/tasks.jsonl",
    "--baseline-outputs-file", "training/heldout_baseline_outputs.jsonl",
    "--prompt-outputs-file", "training/heldout_prompt_outputs.jsonl",
    "--trained-outputs-file", "training/heldout_trained_outputs.jsonl",
    "--bootstrap-iters", "5000",
    "--assume-cost-baseline", str(ASSUME_COST_BASELINE),
    "--assume-cost-prompt", str(ASSUME_COST_PROMPT),
    "--assume-cost-trained", str(ASSUME_COST_TRAINED),
    "--out-ablation", "ablation_results.json",
    "--out-traces", "held_out_traces.jsonl",
]

if WEEK10_RETAIL_SCORE is not None:
    cmd.extend(["--week10-retail-score", str(WEEK10_RETAIL_SCORE)])

subprocess.run(cmd, check=True)
print("Full run complete.")


## Stage 8 - Quick Result Check


In [ ]:
import json
from pathlib import Path

p = Path("ablation_results.json")
assert p.exists(), "ablation_results.json not found"
obj = json.loads(p.read_text(encoding="utf-8"))

keys = [
    "run_mode",
    "variant_summary",
    "delta_a_trained_vs_week10_baseline",
    "delta_b_trained_vs_prompt_only",
    "delta_c_trained_vs_week10_retail",
    "cost_pareto",
]

for k in keys:
    print(f"=== {k} ===")
    print(json.dumps(obj.get(k), indent=2))


## Stage 9 - Final Artifact Check


In [ ]:
from pathlib import Path

required_outputs = [
    "ablation_results.json",
    "held_out_traces.jsonl",
    "training/heldout_trained_outputs.jsonl",
]
for p in required_outputs:
    print(p, "OK" if Path(p).exists() else "MISSING")
